In [1]:
print("AMBAJRI")

AMBAJRI


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import rebound
import assist
import datetime


In [3]:

# -------------------------
# Parameter dasar
# -------------------------
N_CLONES = 2000              ######## JUMLAH KLON
N_KOMP = 3 # jgn diganti (dimensi array data)
T_MAX = 365.25 * 10                      #### HARI
E_sigma= 10                               #### NILAI SIGMA
EARTH_RADIUS_AU = 4.2635e-5
MOON_DISTANCE_AU = 1.1611128e-8

# -------------------------
# TANGGAL AWAL BARU
# -------------------------
date_initial = datetime.datetime(
    2025, 1, 20, tzinfo=datetime.timezone.utc
)

# Julian Date untuk 20 Januari 2025 00:00 UTC
jd_initial = 2460695.5

daperi=24 #data per hari

# -------------------------
# Setup simulasi REBOUND + ASSIST
# -------------------------
sim = rebound.Simulation()

ephem = assist.Ephem(
    "../data/linux_p1550p2650.440",
    "../data/sb441-n16.bsp"
)

extras = assist.Extras(sim, ephem)
sim.t = jd_initial - ephem.jd_ref # Code units will be Julian days relative to jd_ref
# Koreksi relativistik
extras.gr_eih_sources = 11

# Set waktu awal simulasi (hari Julian relatif)
sim.t = jd_initial - ephem.jd_ref

# Mode adaptif IAS15 (penting untuk close encounter)
sim.ri_ias15.adaptive_mode = 2


In [4]:

AU_KM = 149_597_870.7

sim.add(
    "2024 YR4",
    plane="frame",
    date="JD%f" % jd_initial)
mean = np.array([sim.particles[0].x, sim.particles[0].y, sim.particles[0].z, 
                 sim.particles[0].vx, sim.particles[0].vy, sim.particles[0].vz])

sigma_x  =E_sigma *( 65.1004   / AU_KM )
sigma_y  =E_sigma *( 102.8445  / AU_KM)
sigma_z  =E_sigma *( 72.7221   / AU_KM)

sigma_vx = E_sigma *(1.775e-8)
sigma_vy = E_sigma *(3.3573e-8)
sigma_vz = E_sigma *(9.1403e-9)

sigma = np.array([sigma_x, sigma_y, sigma_z, sigma_vx, sigma_vy, sigma_vz])



Searching NASA Horizons for '2024 YR4'... 
Found: (2024 YR4) 


/mnt/d/PENTING KUYLAH/code/lib/python3.12/site-packages/rebound/horizons.py:184: RuntimeWarning: Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.
  warnings.warn("Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.", RuntimeWarning)


In [5]:
data = np.zeros((int(daperi*T_MAX + 1), N_CLONES, N_KOMP))
earth_pos=np.zeros(( int( 1+ (daperi * T_MAX)),3))
moon_pos=np.zeros(( int( 1+ (daperi * T_MAX)),3))
min_dist= np.zeros((int(daperi*T_MAX + 1), N_CLONES, N_KOMP))
# GENERATE t0
# =========================
init = np.random.normal(mean, sigma, size=(N_CLONES, 6))



In [6]:
for i in range(N_CLONES):
    sim.add(
        m=0,
        x=init[i,0],
        y=init[i,1],
        z=init[i,2],
        vx=init[i,3],
        vy=init[i,4],
        vz=init[i,5]
    )


In [7]:
times = np.linspace(sim.t, sim.t + T_MAX, int( 1+ (daperi * T_MAX)))

for i, t in enumerate(times): 
    extras.integrate_or_interpolate(t)
    earth_pos[i,0]=ephem.get_particle("earth", t).x
    earth_pos[i,1]=ephem.get_particle("earth", t).y
    earth_pos[i,2]=ephem.get_particle("earth", t).z
    moon_pos[i,0]=ephem.get_particle("moon", t).x
    moon_pos[i,1]=ephem.get_particle("moon", t).y
    moon_pos[i,2]=ephem.get_particle("moon", t).z
    for j in range(N_CLONES):
        #save data perubahan posisi klon asteroid
        data[i,j,0]=sim.particles[j].x
        data[i,j,1]=sim.particles[j].y
        data[i,j,2]=sim.particles[j].z
        ### calculate dist to sun 
        min_dist[i,j,0]=np.sqrt(sim.particles[j].x**2 + sim.particles[j].y**2 + sim.particles[j].z**2) 
        ### calculate dist to earth
        min_dist[i,j,1]=np.sqrt((sim.particles[j].x - earth_pos[i,0])**2 +(sim.particles[j].y - earth_pos[i,1])**2 +(sim.particles[j].z - earth_pos[i,2])**2)
        ### calculate dist to moon
        min_dist[i,j,2]=np.sqrt((sim.particles[j].x - moon_pos[i,0])**2 +(sim.particles[j].y - moon_pos[i,1])**2 +(sim.particles[j].z - moon_pos[i,2])**2)
        
        

In [8]:
### SAVE FILE DALAN NPY 
np.save("data.npy", data) 
np.save("dist.npy", min_dist)
np.save("earth_pos.npy",earth_pos)
np.save("moon_pos.npy",moon_pos)